# AKTU B.Tech Machine Learning (BOE-073) Assessment Project
## Comparative Study and Empirical Evaluation of Classical ML Algorithms
---
**Student Assessment Notebook**  
**Syllabus Mapping:** Unit II (Decision Trees, Perceptron/ANN), Unit III (Hypothesis Evaluation, Bayesian Learning), Unit IV ($k$-NN)  
**Prescribed Reference:** *Machine Learning* by Tom M. Mitchell (McGraw-Hill)


### Step 1: Ingestion & Exploratory Data Analysis (EDA)
Load Wisconsin Breast Cancer Diagnostic Dataset (569 patient cases, 30 features).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.data_loader import DataLoader

# Initialize loader and export to data/ folder
loader = DataLoader(test_size=0.20, random_state=42)
X, y = loader.load_data(export_csv=True)
stats = loader.get_summary_stats(X, y)

print(f"Total instances: {stats['total_samples']}")
print(f"Feature count: {stats['num_features']}")
print(f"Class Distribution: {stats['class_counts']}")

splits = loader.prepare_splits(X, y, export_splits=True)
print(f"Training set shape: {splits['X_train_scaled'].shape} (Saved to data/train_data.csv)")
print(f"Testing set shape: {splits['X_test_scaled'].shape} (Saved to data/test_data.csv)")

### Step 2: Model Training & Evaluation Across Syllabus Units
* **Unit II:** Decision Tree (Entropy heuristic) & Artificial Neural Network (MLP / Backprop)
* **Unit III:** Gaussian Naive Bayes (Bayesian Learning)
* **Unit IV:** $k$-Nearest Neighbors ($k=5$)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.models_sklearn import get_sklearn_models

models = get_sklearn_models()
results = []

for name, model in models.items():
    model.fit(splits["X_train_scaled"], splits["y_train"])
    preds = model.predict(splits["X_test_scaled"])
    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(splits["y_test"], preds),
        "Precision": precision_score(splits["y_test"], preds),
        "Recall": recall_score(splits["y_test"], preds),
        "F1-Score": f1_score(splits["y_test"], preds)
    })

results_df = pd.DataFrame(results)
results_df

### Step 3: Unit III - Hypothesis Testing & Comparing Learning Algorithms
We evaluate whether performance differences are statistically significant using:
1. **10-Fold Cross-Validation** with Variance & 95% Confidence Intervals
2. **Paired Student's $t$-Tests** across all model pairs ($p$-value threshold $\alpha = 0.05$)

In [ ]:
from src.hypothesis_testing import HypothesisEvaluator

evaluator = HypothesisEvaluator(n_splits=10, random_state=42)
cv_summary, cv_distributions = evaluator.evaluate_cross_validation(models, splits["X_all_scaled"], y)
print("10-Fold Cross-Validation Summary:")
display(cv_summary)

print("\nPairwise Paired t-Tests (Statistical Significance):")
pairwise_df = evaluator.run_all_pairwise_tests(cv_distributions)
display(pairwise_df)

### Step 4: Verification with Pure Python / Scratch Implementations
Testing custom mathematical formulations derived from Tom Mitchell's textbook against Scikit-Learn libraries.

In [ ]:
from src.models_scratch import ScratchKNN, ScratchGaussianNaiveBayes, ScratchPerceptron

# 1. Scratch k-NN
s_knn = ScratchKNN(k=5)
s_knn.fit(splits["X_train_scaled"], splits["y_train"].values)
knn_scratch_pred = s_knn.predict(splits["X_test_scaled"])

# 2. Scratch Naive Bayes
s_nb = ScratchGaussianNaiveBayes()
s_nb.fit(splits["X_train_scaled"], splits["y_train"].values)
nb_scratch_pred = s_nb.predict(splits["X_test_scaled"])

print(f"Scratch k-NN Accuracy: {accuracy_score(splits['y_test'], knn_scratch_pred):.4f}")
print(f"Scratch Naive Bayes Accuracy: {accuracy_score(splits['y_test'], nb_scratch_pred):.4f}")

### Step 5: Visualizations (Saved to outputs/ folder)

In [ ]:
from src.visualizer import Visualizer
vis = Visualizer()
vis.plot_metric_comparison(results_df)
vis.plot_cv_distribution(cv_distributions)
print("All evaluation figures successfully saved in outputs/ directory!")